Data

In [4]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler
from databento import DBNStore

ModuleNotFoundError: No module named 'pandas'

In [ ]:
file = '/content/drive/MyDrive/GLBX-20250327-GP8MUBLVPB/glbx-mdp3-20250227-20250326.ohlcv-1s.dbn.zst'

# Check if the file exists
if not os.path.isfile(file):
    raise FileNotFoundError(f"File not found at: {file}")

df = DBNStore.from_file(file).to_df()
df.index = pd.to_datetime(df.index)
df = df[['open', 'high', 'low', 'close', 'volume']]

scaler = MinMaxScaler()
df[df.columns] = scaler.fit_transform(df[df.columns])

In [1]:
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google'

Feature Engineering

In [ ]:
df['ma5'] = df['close'].rolling(5).mean()
df['ma15'] = df['close'].rolling(15).mean()
df['rsi'] = df['close'].diff().apply(lambda x: max(x, 0)).rolling(14).mean() / df['close'].diff().abs().rolling(14).mean()
ema12 = df['close'].ewm(span=12, adjust=False).mean()
ema26 = df['close'].ewm(span=26, adjust=False).mean()
df['macd'] = ema12 - ema26

df = df.bfill().ffill()

scaler = MinMaxScaler() # Normalize the features at once
df[df.columns] = scaler.fit_transform(df[df.columns])


Split input into patches

In [ ]:
features = ['open', 'high', 'low', 'close', 'volume', 'ma5', 'ma15', 'rsi', 'macd']
X = df[features].values
y = df['close'].shift(-1).fillna(method='ffill').values

N_STEPS = 60
sequence_data = []
for i in range(len(X) - N_STEPS):
    sequence_data.append((X[i:i+N_STEPS], y[i+N_STEPS]))

X_seq = np.array([seq for seq, _ in sequence_data], dtype=np.float32)
y_seq = np.array([target for _, target in sequence_data], dtype=np.float32)

split_idx = int(len(X_seq) * 0.8)
X_train, X_test = X_seq[:split_idx], X_seq[split_idx:]
y_train, y_test = y_seq[:split_idx], y_seq[split_idx:]

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(-1)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).unsqueeze(-1)

class PatchTST(nn.Module):
    def __init__(self, input_len, forecast_len, n_features, patch_len=16, n_heads=4, d_model=64, dropout=0.1):
        super().__init__()
        self.patch_len = patch_len
        self.n_patches = input_len // patch_len
        self.embedding = nn.Linear(patch_len * n_features, d_model)
        self.positional_encoding = nn.Parameter(torch.randn(1, self.n_patches, d_model))
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads, dropout=dropout, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=3)
        self.decoder = nn.Linear(d_model, forecast_len)

    def forward(self, x):
        B, L, C = x.shape
        x = x.view(B, self.n_patches, self.patch_len * C)
        x = self.embedding(x) + self.positional_encoding
        x = self.transformer(x)
        return self.decoder(x[:, -1])



In [ ]:
model = PatchTST(input_len=60, forecast_len=1, n_features=9)
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
EPOCHS = 20

for epoch in range(EPOCHS):
    model.train()
    optimizer.zero_grad()
    preds = model(X_train_tensor)
    loss = loss_fn(preds, y_train_tensor)
    loss.backward()
    optimizer.step()
    print(f"Epoch {epoch+1}/{EPOCHS} - Train Loss: {loss.item():.6f}")

model.eval()
with torch.no_grad():
    test_preds = model(X_test_tensor)
    test_loss = loss_fn(test_preds, y_test_tensor).item()
    print(f"\nTest Loss: {test_loss:.6f}")

    # Predict next close
    latest_seq = torch.tensor(X_seq[-1:], dtype=torch.float32)
    next_pred = model(latest_seq).item()
    predicted_price = scaler.inverse_transform([[next_pred] * len(features)])[0][3]  # index 3 = close
    print(f"Predicted close price (next step): ${predicted_price:.2f}")

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from datasets import load_dataset


INPUT_LEN = 60
FORECAST_LEN = 30
PATCH_LEN = 16
BATCH_SIZE = 64
EPOCHS = 30
LR = 0.001

# Load dataset
REPO_ID = "johnrizzo1/stocks_daily_price"
ds = load_dataset(REPO_ID)
df = ds['train'].to_pandas()
df = df[(df['symbol'] == 'F') & (df['date'] >= '2010-01-01') & (df['date'] <= '2012-12-31')]


df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date')
scaler = MinMaxScaler()
df['close'] = scaler.fit_transform(df[['close']])
data = df['close'].values


class TimeSeriesDataset(Dataset):
    def __init__(self, series, input_len, forecast_len):
        self.series = series
        self.input_len = input_len
        self.forecast_len = forecast_len

    def __len__(self):
        return len(self.series) - self.input_len - self.forecast_len

    def __getitem__(self, idx):
        x = self.series[idx:idx + self.input_len]
        y = self.series[idx + self.input_len:idx + self.input_len + self.forecast_len]
        return torch.tensor(x, dtype=torch.float32).unsqueeze(-1), torch.tensor(y, dtype=torch.float32)

train_dataset = TimeSeriesDataset(data, INPUT_LEN, FORECAST_LEN)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)


class PatchTST(nn.Module):
    def __init__(self, input_len, forecast_len, n_features, patch_len=16, n_heads=4, d_model=64, dropout=0.1):
        super().__init__()
        self.n_patches = input_len // patch_len
        self.patch_len = patch_len
        self.embedding = nn.Linear(patch_len * n_features, d_model)
        self.positional_encoding = nn.Parameter(torch.randn(1, self.n_patches, d_model))
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads, dropout=dropout)
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(d_model * self.n_patches, forecast_len)
        )

    def forward(self, x):
        B, T, C = x.shape
        x = x.view(B, self.n_patches, self.patch_len * C)
        x = self.embedding(x) + self.positional_encoding[:, :self.n_patches]
        x = self.encoder(x)
        out = self.head(x)
        return out

model = PatchTST(INPUT_LEN, FORECAST_LEN, n_features=1, patch_len=PATCH_LEN)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.MSELoss()


for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0
    for xb, yb in train_loader:
        optimizer.zero_grad()
        pred = model(xb)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {epoch_loss / len(train_loader):.4f}")


model.eval()
with torch.no_grad():
    last_seq = torch.tensor(data[-INPUT_LEN:], dtype=torch.float32).unsqueeze(0).unsqueeze(-1)  # shape (1, T, 1)
    forecast = model(last_seq).squeeze().numpy()


forecast_real = scaler.inverse_transform(forecast.reshape(-1, 1)).flatten()
past_real = scaler.inverse_transform(data[-INPUT_LEN:].reshape(-1, 1)).flatten()


plt.figure(figsize=(10, 5))
plt.plot(range(INPUT_LEN), past_real, label="Past")
plt.plot(range(INPUT_LEN, INPUT_LEN + FORECAST_LEN), forecast_real, label="Forecast", color='orange')
plt.title("PatchTST: 30-day Stock Price Forecast")
plt.xlabel("Time")
plt.ylabel("Price")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()